# Problem Analysis Workshop C — Capital Bikeshare

**Group 7**

1. Sukhchain Singh (9111541)
2. Preethi (9125985)
3. Arya (9084843)

## Step 1 — Define the research question

**How well do weather and calendar variables predict daily Capital Bikeshare rentals in Washington, D.C.? Does multiple linear regression predict rentals more accurately than either of the two simple linear regression models?**

Our **target variable** is `cnt`, which is the total number of bikes rented on a given day. A **predictor** is a piece of information we use to estimate that total. Temperature, rainfall, and the day of the week are examples of predictors.

A **simple linear regression** uses one predictor at a time. Our team will compare two simple models, then use **multiple linear regression** to see whether combining several predictors improves accuracy. In my section, I prepare and check the data that the team will use for all three models.

## Step 2 — Choose and load the datasets

We chose the **Capital Bikeshare daily dataset** (`day.csv`) because it contains the outcome we want to predict, the total number of rentals per day. It also includes calendar details such as the weekday and whether it was a holiday, along with basic weather measurements.

We added **Open-Meteo historical weather data** (`open_meteo_weather.csv`) for Washington, D.C. It gives us more detailed measurements for the same period. For example, the bike dataset gives us a broad weather category, while Open-Meteo tells us how much rain fell and how long precipitation lasted. These details let us test whether more specific weather measurements help predict rental demand.

Both files cover January 1, 2011, through December 31, 2012, with one row per day. We load the files from the project's `data` folder and preview them to confirm their size and identify the date column in each file.

[UCI Bike Sharing dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)  
[Open-Meteo Historical Weather documentation](https://open-meteo.com/en/docs/historical-weather-api)

In [4]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Look for the CSVs in the current folder's data folder,
# then check each parent folder.
current_folder = Path.cwd()

for folder in [current_folder, *current_folder.parents]:
    data_folder = folder / "data"
    if (data_folder / "day.csv").is_file() and (
        data_folder / "open_meteo_weather.csv"
    ).is_file():
        break
else:
    raise FileNotFoundError(f"Could not find both CSVs in a data folder above {current_folder}")

bike = pd.read_csv(data_folder / "day.csv")
weather = pd.read_csv(data_folder / "open_meteo_weather.csv", skiprows=3)

# Remove units from weather headers, such as "rain_sum (mm)".
weather.columns = weather.columns.str.strip().str.replace(
    r"\s+\([^)]*\)$", "", regex=True
)

print("Loading files from:", data_folder)

print("Bike dataset (rows, columns):", bike.shape)
display(bike.head(3))

print("Weather dataset (rows, columns):", weather.shape)
display(weather.head(3))

Loading files from: c:\Users\aarya\Desktop\DA\Data_Analysis_Presentation1\data
Bike dataset (rows, columns): (731, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349


Weather dataset (rows, columns): (731, 20)


,time,rain_sum,snowfall_sum,precipitation_hours,shortwave_radiation_sum,daylight_duration,sunshine_duration,wind_gusts_10m_max,temperature_2m_max,temperature_2m_min,cloud_cover_mean,pressure_msl_mean,pressure_msl_max,pressure_msl_min,relative_humidity_2m_min,dew_point_2m_mean,wind_direction_10m_dominant,cloud_cover_max,cloud_cover_min,wind_speed_10m_max
0,2011-01-01,2.7,0.0,6.0,5.59,34147.85,14400.00,41.4,14.8,1.7,85,1018.5,1021.2,1015.6,69,6.0,197,100,0,17.7
1,2011-01-02,2.8,0.0,8.0,1.25,34189.21,0.00,42.5,14.6,0.5,78,1016.3,1020.2,1013.9,35,2.5,304,100,0,18.9
2,2011-01-03,0.0,0.0,0.0,10.22,34234.59,33210.34,37.4,3.5,-3.6,6,1023.8,1026.7,1020.7,34,-10.4,296,89,0,17.4


## Step 3 — Merge the datasets by date

The bike dataset stores its date in `dteday`, and the weather dataset stores it in `time`. We rename both columns to `date` and convert them to the same data type. The date becomes the **join key**, which is the field pandas uses to match records. After the merge, one row contains both the rental count and the weather measurements for that day.

We use a **left join** with the bike dataset on the left. This keeps every bike rental day and lets us see whether any of those dates are missing a weather record. We also use a **one-to-one check** because each date should appear only once in each file. This prevents duplicate weather records from creating extra rental rows.

The merge kept **731 daily records**, with **731 matched dates and 0 unmatched dates**. Every bike rental day has a corresponding weather record.

In [6]:
# Give both date columns the same name.
bike_by_date = bike.rename(columns={"dteday": "date"})
weather_by_date = weather.rename(columns={"time": "date"})

# Convert the date columns from text to the same pandas date type.
bike_by_date["date"] = pd.to_datetime(bike_by_date["date"], format="%Y-%m-%d")
weather_by_date["date"] = pd.to_datetime(weather_by_date["date"], format="%Y-%m-%d")

# Check that every row has a date before matching records.
assert bike_by_date["date"].notna().all(), "The bike dataset has missing dates."
assert weather_by_date["date"].notna().all(), "The weather dataset has missing dates."

# Keep all bike rental days and attach the weather for the same date.
merged = bike_by_date.merge(
    weather_by_date,
    on="date",
    how="left",
    validate="one_to_one",
    indicator=True
)

# The temporary _merge column shows which bike days found a weather match.
matched_days = merged["_merge"].eq("both").sum()
unmatched_days = merged["_merge"].eq("left_only").sum()

print("Rows after merging:", len(merged))
print("Days with a weather match:", matched_days)
print("Days without a weather match:", unmatched_days)

assert len(merged) == 731, "Expected 731 daily records after merging."
assert unmatched_days == 0, "Some bike rental days are missing a weather record."

# Remove the temporary column and preview the combined data.
merged = merged.drop(columns="_merge")
display(merged[["date", "cnt", "temp", "rain_sum", "sunshine_duration"]].head())

Rows after merging: 731
Days with a weather match: 731
Days without a weather match: 0


,date,cnt,temp,rain_sum,sunshine_duration
0,2011-01-01,985,0.344167,2.7,14400.00
1,2011-01-02,801,0.363478,2.8,0.00
2,2011-01-03,1349,0.196364,0.0,33210.34
3,2011-01-04,1562,0.200000,0.0,32026.83
4,2011-01-05,1600,0.226957,0.0,33310.79


## Step 4 — Transform the weather variables

We create three **daily ranges** for temperature, cloud cover, and sea-level pressure. We calculate each range by subtracting the daily minimum from the daily maximum. For example, if the temperature went from 10°C to 18°C, the range was 8°C.

The range gives us one number that describes how much a measurement varied during the day. We can use it to test whether days with larger weather changes tend to have different rental totals.

We also convert wind direction from degrees into eight **compass categories**, such as North, Northeast, and East. Wind angles wrap around at 360°, so 359° and 1° both point almost north. Grouping them puts both angles in the same category and allows the model to compare wind directions as groups.

In [7]:
# Make a copy of the merged data before adding new columns.
prepared = merged.copy()

# Calculate each daily range as maximum minus minimum.
prepared["temp_range"] = (
    prepared["temperature_2m_max"] - prepared["temperature_2m_min"]
)
prepared["cloud_range"] = (
    prepared["cloud_cover_max"] - prepared["cloud_cover_min"]
)
prepared["pressure_range"] = (
    prepared["pressure_msl_max"] - prepared["pressure_msl_min"]
)

# Shift the angle so values near 0° and 360° fall in the North group.
wind_angle = (prepared["wind_direction_10m_dominant"] + 22.5) % 360

prepared["wind_direction"] = pd.cut(
    wind_angle,
    bins=range(0, 361, 45),
    labels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"],
    right=False,
    ordered=False
)

# Preview the original wind angle and the four new predictors.
preview_columns = [
    "date", "temp_range", "cloud_range", "pressure_range",
    "wind_direction_10m_dominant", "wind_direction"
]
display(prepared[preview_columns].head())

,date,temp_range,cloud_range,pressure_range,wind_direction_10m_dominant,wind_direction
0,2011-01-01,13.1,100,5.6,197,S
1,2011-01-02,14.1,100,6.3,304,NW
2,2011-01-03,7.1,89,6.0,296,NW
3,2011-01-04,11.6,97,11.0,218,SW
4,2011-01-05,7.4,88,5.7,317,NW


## Step 5 — Select predictors and set data types

We select **25 predictors**, with nine from Capital Bikeshare and 16 from Open-Meteo. We count each categorical variable once before the regression model turns its levels into indicator columns. We keep `date` to identify the records and `cnt` as the target.

We exclude the following fields from the predictor list:

- `instant` is a row ID. We already have `date` to identify each record.
- `casual` and `registered` add up to `cnt`. Including them would cause **target leakage** because the model could use those two values to recover the answer directly.
- `atemp` measures feels-like temperature and is strongly correlated with `temp`. We keep `temp` to avoid using two very similar measurements.
- `workingday` can be determined from `weekday` and `holiday`, which are already included.

We convert season, month, weekday, weather condition, and wind direction to **categorical variables** because their values represent groups. For example, a weekday code of 6 is a label for a particular day of the week. We keep `yr` and `holiday` as **binary indicators**, with 0 and 1 representing the two groups.

In [8]:
bike_predictors = [
    "season", "yr", "mnth", "holiday", "weekday",
    "weathersit", "temp", "hum", "windspeed"
]

weather_predictors = [
    "rain_sum", "snowfall_sum", "precipitation_hours",
    "shortwave_radiation_sum", "daylight_duration", "sunshine_duration",
    "wind_gusts_10m_max", "wind_speed_10m_max", "wind_direction",
    "temp_range", "cloud_cover_mean", "cloud_range",
    "pressure_msl_mean", "pressure_range",
    "relative_humidity_2m_min", "dew_point_2m_mean"
]

target = "cnt"
predictors = bike_predictors + weather_predictors

# Keep the date, target, and selected predictors.
study_data = prepared[["date", target] + predictors].copy()

# Mark columns that represent groups.
categorical_columns = ["season", "mnth", "weekday", "weathersit", "wind_direction"]

for column in categorical_columns:
    study_data[column] = study_data[column].astype("category")

print("Bike predictors:", len(bike_predictors))
print("Weather predictors:", len(weather_predictors))
print("Total predictors:", len(predictors))
print("Dataset shape:", study_data.shape)

display(study_data.head(3))

Bike predictors: 9
Weather predictors: 16
Total predictors: 25
Dataset shape: (731, 27)


,date,cnt,season,yr,mnth,holiday,weekday,weathersit,temp,hum,...,wind_gusts_10m_max,wind_speed_10m_max,wind_direction,temp_range,cloud_cover_mean,cloud_range,pressure_msl_mean,pressure_range,relative_humidity_2m_min,dew_point_2m_mean
0,2011-01-01,985,1,0,1,0,6,2,0.344167,0.805833,...,41.4,17.7,S,13.1,85,100,1018.5,5.6,69,6.0
1,2011-01-02,801,1,0,1,0,0,2,0.363478,0.696087,...,42.5,18.9,NW,14.1,78,100,1016.3,6.3,35,2.5
2,2011-01-03,1349,1,0,1,0,1,1,0.196364,0.437273,...,37.4,17.4,NW,7.1,6,89,1023.8,6.0,34,-10.4


## Step 6 — Verify the final dataset

Our final dataframe, `study_data`, contains **731 unique days and 27 columns**. The columns include 25 predictors, the date, and the target `cnt`. We found **zero missing values and zero duplicate dates**.

We also checked for **constant predictors**. A constant predictor has the same value every day, so it cannot help explain why rental totals differ. All 25 predictors had more than one value. We then reviewed summary statistics for rentals, temperature, rainfall, and sunshine to check their typical values and spot any unexpected extremes.

Member 2 will use `study_data` to create the training and test sets. Member 3 will use the same split so the team can compare the prediction errors of all models on the same test days.

In [9]:
missing_count = study_data.isna().sum().sum()
duplicate_dates = study_data["date"].duplicated().sum()

constant_predictors = [
    column for column in predictors
    if study_data[column].nunique() < 2
]

print("Daily records:", len(study_data))
print("Duplicate dates:", duplicate_dates)
print("Missing values:", missing_count)
print("Predictors:", len(predictors))
print("Constant predictors:", constant_predictors)
print("Final dataset shape:", study_data.shape)

assert len(study_data) == 731, "Expected 731 daily records."
assert duplicate_dates == 0, "Duplicate dates found."
assert missing_count == 0, "Missing values found."
assert len(predictors) == len(set(predictors)) == 25, "Expected 25 distinct predictors."
assert not constant_predictors, "At least one predictor has no variation."

summary_columns = ["cnt", "temp", "rain_sum", "sunshine_duration"]
display(study_data[summary_columns].describe().round(2))

Daily records: 731
Duplicate dates: 0
Missing values: 0
Predictors: 25
Constant predictors: []
Final dataset shape: (731, 27)


,cnt,temp,rain_sum,sunshine_duration
count,731.00,731.00,731.00,731.00
mean,4504.35,0.50,2.80,32924.07
std,1937.21,0.18,7.29,14265.83
min,22.00,0.06,0.00,0.00
25%,3152.00,0.34,0.00,25409.66
50%,4548.00,0.50,0.00,35679.92
75%,5956.00,0.66,2.05,43200.00
max,8714.00,0.86,85.00,51107.74
